<a href="https://colab.research.google.com/github/div652/GPT-2/blob/main/play_gpt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Interact with GPT2

Using this notebook to verify if the model reproduced by my own implementation is exactly the same or not.

In [3]:
#sample from the model
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model", max_new_tokens=30, num_return_sequences=5)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hello, I\'m a language model for you" —Miles Teller (@milesteller) October 13, 2015\n\nI guess I\'ll just have to watch out for'},
 {'generated_text': "Hello, I'm a language model. I'm not a designer. I'm just an engineer.\n\nI'm a programmer. I'm smart. I'm a writer. I"},
 {'generated_text': 'Hello, I\'m a language model for this.\n\nYou have probably already seen my tutorial on the topic of how to use the language. However, what is "language model"?'},
 {'generated_text': "Hello, I'm a language modeler. I've been writing for some time now and I've seen a lot of very interesting things in the programming world as well. I've done"},
 {'generated_text': 'Hello, I\'m a language modeler, and I\'m going to write a Python script for you.\n\nI am working on a website called "PythonScript" that will allow'}]

## Tokeniser

In [6]:
#return a torch tensor of prompt to tokens on cpu.
import tiktoken
def get_tokens(prompt):
  enc = tiktoken.get_encoding('gpt2')
  tokens = enc.encode(prompt)
  tokens = torch.tensor(tokens, dtype = torch.long)
  return tokens
def get_text(tokens):
  enc = tiktoken.get_encoding('gpt2')
  decoded = enc.decode(tokens)
  return decoded


## Generation using Manual sampling.

In [9]:
import torch
import torch.nn.functional as F

from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained('gpt2')
print("Hugging Face model loaded successfully")

num_return_sequences = 5
max_length = 30
model.eval() # this removes things like dropouts
model.to('cuda')
tokens = get_tokens("Hello, I'm a language model")
tokens = tokens.unsqueeze(0).repeat(num_return_sequences,1) # repeat along the dim1
x = tokens.to('cuda')
torch.manual_seed(42)
torch.cuda.manual_seed(42)
print(x.shape)
while x.size(1) < max_length:
  with torch.no_grad():
    logits = model(x)[0] # B,T,vocab_size
    logits = logits[:,-1,:] # B,vocab_size. Only need the last logits
    probs = F.softmax(logits,dim=-1)
    # we pick only top50, so that hte model doesn't stray away into unkonwn zones. and styas in "good" zones
    topk_probs, topk_indices = torch.topk(probs,50,dim=-1) # this is the temperature sampling essentially. we don't pick the most probable, rather we sample from the distribution
    ix = torch.multinomial(topk_probs,1)#B,1
    xcol = torch.gather(topk_indices,-1,ix)
    x = torch.cat((x,xcol),dim=1)

  # print the generated text
for i in range(num_return_sequences):
  tokens = x[i,:max_length].tolist()
  print(">",get_text(tokens))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hugging Face model loaded successfully
torch.Size([5, 7])
> Hello, I'm a language modeler. In fact I use that name from C# to describe our development and development workflow. We have many different
> Hello, I'm a language model. Language models are not that complicated unless you know to change the behavior of some of your functions, of the variable
> Hello, I'm a language model. I make the syntax based on it. For example, it can be used to define the syntax of a language
> Hello, I'm a language modeler - I like to do a lot of abstraction and I've also seen the ability to generate generic programming languages like
> Hello, I'm a language model that has a language model in the domain of computer biology. If you study computer systems in your undergraduate degree or at


Printing model state dictionary.

In [21]:
sd_hf = model.state_dict()
for k,v in sd_hf.items():
  print(k,v.shape)

transformer.wte.weight torch.Size([50257, 768])
transformer.wpe.weight torch.Size([1024, 768])
transformer.h.0.ln_1.weight torch.Size([768])
transformer.h.0.ln_1.bias torch.Size([768])
transformer.h.0.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.0.attn.c_attn.bias torch.Size([2304])
transformer.h.0.attn.c_proj.weight torch.Size([768, 768])
transformer.h.0.attn.c_proj.bias torch.Size([768])
transformer.h.0.ln_2.weight torch.Size([768])
transformer.h.0.ln_2.bias torch.Size([768])
transformer.h.0.mlp.c_fc.weight torch.Size([768, 3072])
transformer.h.0.mlp.c_fc.bias torch.Size([3072])
transformer.h.0.mlp.c_proj.weight torch.Size([3072, 768])
transformer.h.0.mlp.c_proj.bias torch.Size([768])
transformer.h.1.ln_1.weight torch.Size([768])
transformer.h.1.ln_1.bias torch.Size([768])
transformer.h.1.attn.c_attn.weight torch.Size([768, 2304])
transformer.h.1.attn.c_attn.bias torch.Size([2304])
transformer.h.1.attn.c_proj.weight torch.Size([768, 768])
transformer.h.1.attn.c_proj.bias 